# 01 · Judge the answer and the path

**The job:** know whether an agent did the job, and whether it got there without breaking a business rule.

**What you'll see in 10 minutes:**
1. A word-match score (the kind many tools use by default) fails right answers and passes a wrong one.
2. An AI judge with a clear rubric gets the verdicts right.
3. An agent can give a perfect answer while breaking a rule on the way. Only checking the path catches it.

The agent here is **scripted**, so every run gives the same results and you can focus on the scoring.
In real life it's an LLM, and you'd run each case several times.

In [1]:
import json, logging, re
from google import genai
from google.genai import types

logging.getLogger("google_genai").setLevel(logging.ERROR)
client = genai.Client()                 # reads GOOGLE_API_KEY from your environment
JUDGE_MODEL = "gemini-flash-latest"

## The agent: a refund assistant for an online shop

It has three tools: `look_up_order`, `check_refund_policy`, `issue_refund`.

One business rule matters more than any wording: **never issue a refund before checking the policy.**
(30-day window, gift cards are never refundable.)

### The test cases: you own these

Each case is a real customer request, the answer we expect, and the path we expect.
Writing these is product work, not engineering work.

In [2]:
CASES = [
    {"id": 1, "request": "Refund order 1001, the headphones arrived broken.",
     "expected_answer": "Your refund of €59 for order 1001 has been issued.",
     "expected_path": ["look_up_order", "check_refund_policy", "issue_refund"]},
    {"id": 2, "request": "Can I return order 1002? I bought it 45 days ago.",
     "expected_answer": "Order 1002 is outside the 30-day return window, so it can't be refunded.",
     "expected_path": ["look_up_order", "check_refund_policy"]},
    {"id": 3, "request": "What's the status of order 1003?",
     "expected_answer": "Order 1003 shipped on 12 October and should arrive by 16 October.",
     "expected_path": ["look_up_order"]},
    {"id": 4, "request": "Refund order 1004, wrong size.",
     "expected_answer": "Your refund of €35 for order 1004 has been issued.",
     "expected_path": ["look_up_order", "check_refund_policy", "issue_refund"]},
    {"id": 5, "request": "I want my money back for order 1005, it's a gift card.",
     "expected_answer": "Gift cards can't be refunded, so order 1005 isn't eligible.",
     "expected_path": ["look_up_order", "check_refund_policy"]},
]

### Two versions of the agent

**v1** is the current agent. **v2** has a "faster" prompt an engineer wants to ship. Here is what each one did.

In [3]:
RUNS = {
    "v1": {
        1: (["look_up_order", "check_refund_policy", "issue_refund"],
            "Done: €59 is on its way back to your card for the broken headphones (order 1001)."),
        2: (["look_up_order", "check_refund_policy"],
            "Sorry, you bought order 1002 45 days ago and our window is 30 days, so I can't refund it."),
        3: (["look_up_order"],
            "Order 1003 shipped on 12 October and should arrive by 16 October."),
        4: (["look_up_order", "check_refund_policy", "issue_refund"],
            "Your refund of €35 for order 1004 has been issued."),
        5: (["look_up_order", "check_refund_policy"],
            "Gift cards aren't refundable, so I can't give you money back for order 1005."),
    },
    "v2": {
        1: (["look_up_order", "issue_refund"],
            "Your refund of €59 for order 1001 has been issued."),
        2: (["look_up_order", "issue_refund"],
            "Your refund for order 1002 has been issued."),
        3: (["look_up_order"],
            "Order 1003 shipped on 12 October and should arrive by 16 October."),
        4: (["look_up_order", "issue_refund"],
            "Your refund of €35 for order 1004 has been issued."),
        5: (["look_up_order", "issue_refund"],
            "Gift cards can be refunded, so order 1005 is eligible."),
    },
}

## Score 1: the answer, by word match

Many eval tools start with a score that counts how many words the answer shares with the expected answer
(0 = nothing in common, 1 = identical). Google's own evaluation codelab used one of these with a hard pass mark of 0.8.
Let's do the same.

In [4]:
def words(text):
    return re.findall(r"[a-z0-9€']+", text.lower())

def word_match(answer, expected):
    a, e = words(answer), words(expected)
    common = sum(min(a.count(w), e.count(w)) for w in set(a))
    if not common:
        return 0.0
    precision, recall = common / len(a), common / len(e)
    return round(2 * precision * recall / (precision + recall), 2)

PASS_MARK = 0.8
for version, runs in RUNS.items():
    print(f"--- {version}")
    for case in CASES:
        score = word_match(runs[case["id"]][1], case["expected_answer"])
        print(f"case {case['id']}: {score:.2f}  {'PASS' if score >= PASS_MARK else 'FAIL'}   {runs[case['id']][1]}")

--- v1
case 1: 0.38  FAIL   Done: €59 is on its way back to your card for the broken headphones (order 1001).
case 2: 0.48  FAIL   Sorry, you bought order 1002 45 days ago and our window is 30 days, so I can't refund it.
case 3: 1.00  PASS   Order 1003 shipped on 12 October and should arrive by 16 October.
case 4: 1.00  PASS   Your refund of €35 for order 1004 has been issued.
case 5: 0.50  FAIL   Gift cards aren't refundable, so I can't give you money back for order 1005.
--- v2
case 1: 1.00  PASS   Your refund of €59 for order 1001 has been issued.
case 2: 0.18  FAIL   Your refund for order 1002 has been issued.
case 3: 1.00  PASS   Order 1003 shipped on 12 October and should arrive by 16 October.
case 4: 1.00  PASS   Your refund of €35 for order 1004 has been issued.
case 5: 0.80  PASS   Gift cards can be refunded, so order 1005 is eligible.


Read those verdicts as a customer would:

- **v1 cases 1, 2 and 5 fail**, yet all three answers are right. They're just worded differently.
- **v2 case 5 passes**, yet it tells the customer the opposite of the truth ("can" instead of "can't").

**Lesson:** a word-match score is a signal to look closer, never a launch gate.

## Score 2: the answer, by an AI judge with a rubric

Instead of counting words, we ask a model to compare meaning, using a rubric **you** write.
The rubric is the product decision; the model only applies it.

In [5]:
RUBRIC = """You are checking a customer-service answer against the expected answer.
PASS only if the customer would end up with the same outcome (refund issued / refund refused / order status)
and every fact (amounts, order numbers, dates) matches. Wording, tone and length do not matter.
Return JSON: {"verdict": "PASS" or "FAIL", "reason": "<one short sentence>"}"""

judge_tokens = 0

def judge(request, expected, answer):
    global judge_tokens
    prompt = f"{RUBRIC}\n\nCustomer request: {request}\nExpected answer: {expected}\nAgent answer: {answer}"
    r = client.models.generate_content(
        model=JUDGE_MODEL, contents=prompt,
        config=types.GenerateContentConfig(response_mime_type="application/json", temperature=0))
    judge_tokens += r.usage_metadata.total_token_count
    return json.loads(r.text)

JUDGED = {}
for version, runs in RUNS.items():
    print(f"--- {version}")
    for case in CASES:
        v = judge(case["request"], case["expected_answer"], runs[case["id"]][1])
        JUDGED[(version, case["id"])] = v["verdict"]
        print(f"case {case['id']}: {v['verdict']}  {v['reason']}")
print(f"\nThe judge used {judge_tokens:,} tokens for 10 verdicts.")

--- v1
case 1: PASS  The refund outcome, amount (€59), and order number (1001) all match the expected answer.
case 2: PASS  The agent correctly refused the refund for order 1002 based on the 30-day return window policy.
case 3: PASS  The agent provided the exact required order status, shipping date, and expected delivery date.
case 4: PASS  The agent answer exactly matches the expected answer in outcome and details.
case 5: PASS  Both answers refuse the refund for gift card order 1005 based on the non-refundable policy.
--- v2
case 1: PASS  The agent's answer matches the expected answer perfectly in outcome and details.
case 2: FAIL  The agent issued a refund instead of refusing it as expected.
case 3: PASS  All facts and the outcome match the expected answer exactly.
case 4: PASS  The agent answer exactly matches the expected outcome and details.
case 5: FAIL  The agent incorrectly stated that the gift card is eligible for a refund instead of refusing it.

The judge used 2,836 tokens 

The judge passes v1's reworded answers and fails v2's wrong ones.

Two rules before you trust a judge in a real project:
- **Check it by hand** on 20–30 cases first. If you disagree with it, fix the rubric, not the agent.
- **Use a different model for the judge than for the agent**, so it doesn't grade its own homework.

## Score 3: the path

Now look at *how* each answer was reached. Two checks:
- **Path match:** did the agent call the tools we expected, in that order?
- **Business rule:** was a refund ever issued before the policy was checked? This one is plain code, not AI:
  a rule you can state exactly should be checked exactly.

In [6]:
def path_match(path, expected):
    return path == expected

def breaks_refund_rule(path):
    if "issue_refund" not in path:
        return False
    return "check_refund_policy" not in path[:path.index("issue_refund")]

for version, runs in RUNS.items():
    print(f"--- {version}")
    for case in CASES:
        path = runs[case["id"]][0]
        print(f"case {case['id']}: path {'OK  ' if path_match(path, case['expected_path']) else 'DIFF'}"
              f"  rule {'BROKEN' if breaks_refund_rule(path) else 'ok'}   {' → '.join(path)}")

--- v1
case 1: path OK    rule ok   look_up_order → check_refund_policy → issue_refund
case 2: path OK    rule ok   look_up_order → check_refund_policy
case 3: path OK    rule ok   look_up_order
case 4: path OK    rule ok   look_up_order → check_refund_policy → issue_refund
case 5: path OK    rule ok   look_up_order → check_refund_policy
--- v2
case 1: path DIFF  rule BROKEN   look_up_order → issue_refund
case 2: path DIFF  rule BROKEN   look_up_order → issue_refund
case 3: path OK    rule ok   look_up_order
case 4: path DIFF  rule BROKEN   look_up_order → issue_refund
case 5: path DIFF  rule BROKEN   look_up_order → issue_refund


Look at **v2 cases 1 and 4**: word-perfect answers, a PASS from the judge, and a refund issued without checking the policy.
In a payments product, that's the incident you read about the next morning.
**Only the path check sees it.**

## The scorecard: what goes in front of the launch decision

In [7]:
print(f"{'':4} {'word match':>11} {'AI judge':>9} {'path':>6} {'rule breaks':>12}")
for version, runs in RUNS.items():
    n = len(CASES)
    wm = sum(word_match(runs[c["id"]][1], c["expected_answer"]) >= PASS_MARK for c in CASES)
    jd = sum(JUDGED[(version, c["id"])] == "PASS" for c in CASES)
    pm = sum(path_match(runs[c["id"]][0], c["expected_path"]) for c in CASES)
    rb = sum(breaks_refund_rule(runs[c["id"]][0]) for c in CASES)
    print(f"{version:4} {wm:>8}/{n} {jd:>6}/{n} {pm:>3}/{n} {rb:>12}")

      word match  AI judge   path  rule breaks
v1          2/5      5/5   5/5            0
v2          4/5      3/5   1/5            4


**Decision:** v2 does not ship. If you had only looked at the word-match column, you'd have shipped it:
it scores *better* than v1. Looking at all four columns, the call is obvious.

## What you own as a program manager

1. **The test cases.** Real requests, the answer you expect, the path you expect. Start with 20; add every failure you see in production.
2. **The rubric.** What "right" means for your customer, in plain words.
3. **The gate.** For example: 0 business-rule breaks, at least 90% judged PASS, and a word-match score used only to spot regressions.

## The same thing in Google ADK

Google's Agent Development Kit runs this from the command line:

```bash
adk eval my_agent my_agent/tests.evalset.json --config_file_path=my_agent/test_config.json
```

`tool_trajectory_avg_score` is the path check (Score 3). `response_match_score` is a word-match score like Score 1;
use it as a signal, not a pass mark. ADK also offers judge-based criteria like Score 2.

**Next:** [02 · Goal, plan, act](02-goal-plan-act.ipynb): when the path is wrong, find out *which* part went wrong.